# Causal Inference in Practice
## Week 13 — Heterogeneous Effects & Policy Learning · Practice Notebook

> **Block IV — Modern methods & application**
>
> Move from "does it work on average?" to "for whom — and what should we do about it?"

**How to use this notebook.** Run the cells top to bottom. Sections marked
**🔧 Exercise** contain a `# TODO` for you to complete; a matching
**✅ Solution** cell follows (collapsed in spirit — try it yourself first).
Every dataset here is *simulated with a known ground truth*, so you can always
check whether your estimate recovered the right answer.

*Estimated time: 60–90 minutes. Toolkit: `numpy`, `pandas`, `statsmodels`,
`scikit-learn`, `matplotlib` — all standard.*

---


In [ ]:
# --- Environment check & shared setup -------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (7, 4.2), "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 11})

RNG = np.random.default_rng(7)   # one seed for the whole notebook → reproducible
print("Environment OK — numpy", np.__version__, "| pandas", pd.__version__)

## 1 · The ATE can hide who is helped and who is harmed

We simulate a world with a **known, covariate-varying** treatment effect τ(x) = E[Y(1) − Y(0) | X = x]. Because we built it, we can always compare any estimate to the truth. Treatment `W` is *randomized* (so confounding is not the issue here) — the whole story this week is **heterogeneity**, not bias.

We deliberately center τ(x) near zero: the **ATE is small**, yet a real fraction of units are actually *harmed* (τ < 0). That is exactly the situation where the average is the wrong thing to report.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

def make_data(n):
    """Randomized W; true CATE tau(x) varies with X0 and X1."""
    X = RNG.uniform(0, 1, size=(n, 5))
    tau = 2.5 * X[:, 0] - 1.5 * X[:, 1] - 0.6     # KNOWN ground-truth CATE
    W = RNG.binomial(1, 0.5, size=n)              # randomized treatment
    base = 2.0 * X[:, 2] + X[:, 3] + RNG.normal(0, 1.0, size=n)
    Y = base + W * tau                            # only W*tau is causal
    return X, W, Y, tau

# A train split (to fit models) and a test split (to evaluate honestly)
Xtr, Wtr, Ytr, tau_tr = make_data(4000)
Xte, Wte, Yte, tau_te = make_data(4000)

ate_true = tau_te.mean()
ate_dim  = Yte[Wte == 1].mean() - Yte[Wte == 0].mean()  # diff-in-means
print(f'True ATE            = {ate_true:+.3f}')
print(f'Estimated ATE (DiM) = {ate_dim:+.3f}   (randomized, so unbiased)')
print(f'SD of true CATE     = {tau_te.std():.3f}')
print(f'Range of true CATE  = [{tau_te.min():+.2f}, {tau_te.max():+.2f}]')
print(f'Fraction HARMED (tau<0) = {(tau_te < 0).mean():.1%}')

assert abs(ate_dim - ate_true) < 0.15, 'diff-in-means should recover the ATE'
assert tau_te.std() > 0.5, 'we want genuinely heterogeneous effects'

The ATE is near zero and the difference-in-means recovers it — yet **over half the population is harmed** while the rest benefit. Reporting only the ATE here would tell you to abandon a treatment that is excellent for an identifiable subgroup. Let's find that subgroup by estimating τ(x).

In [ ]:
# Quick look: the true effect clearly varies with X0 (and X1).
fig, ax = plt.subplots()
sc = ax.scatter(Xte[:, 0], tau_te, c=Xte[:, 1], s=6, cmap='viridis')
ax.axhline(0, color='k', lw=1)
ax.axhline(ate_true, color='crimson', ls='--', lw=1.5, label='ATE')
ax.set_xlabel('X0'); ax.set_ylabel('true CATE  tau(x)')
ax.set_title('A flat ATE (red) hides a strong gradient in tau(x)')
ax.legend(); fig.colorbar(sc, label='X1')
print('Units above 0 benefit; units below 0 are harmed.')

## 2 · Two meta-learners: T-learner and S-learner

A **meta-learner** turns ordinary regressors into a CATE estimator.

- **T-learner (Two models):** fit one forest on the treated, one on the controls; τ̂(x) = μ̂₁(x) − μ̂₀(x).
- **S-learner (Single model):** fit one forest with `W` appended as a feature; τ̂(x) = μ̂(x, 1) − μ̂(x, 0).

We keep the forests small for speed and fit on the **train** split, predict on the **test** split.

In [ ]:
# Small forests keep this fast; same settings for every learner.
RF = dict(n_estimators=200, max_depth=8, min_samples_leaf=20,
          random_state=0)

# ---- T-learner: two separate forests ----
m1 = RandomForestRegressor(**RF).fit(Xtr[Wtr == 1], Ytr[Wtr == 1])
m0 = RandomForestRegressor(**RF).fit(Xtr[Wtr == 0], Ytr[Wtr == 0])
cate_T = m1.predict(Xte) - m0.predict(Xte)

# ---- S-learner: one forest with W as a feature ----
mS = RandomForestRegressor(**RF).fit(np.column_stack([Xtr, Wtr]), Ytr)
cate_S = (mS.predict(np.column_stack([Xte, np.ones(len(Xte))])) -
          mS.predict(np.column_stack([Xte, np.zeros(len(Xte))])))

print('Fitted T-learner (2 forests) and S-learner (1 forest).')
print('Predicted CATE for first 5 test units (T-learner):')
print(np.round(cate_T[:5], 3))

Because the data are simulated we can do something impossible in real life: **score the CATE estimates against the truth.** We report the correlation with τ(x) and the MSE, and assert that the T-learner tracks the truth strongly.

In [ ]:
def score(name, cate):
    r   = np.corrcoef(cate, tau_te)[0, 1]
    mse = np.mean((cate - tau_te) ** 2)
    print(f'{name:>10}-learner:  corr(tau_hat, tau_true) = {r:.3f}   '
          f'MSE = {mse:.3f}')
    return r, mse

rT, mseT = score('T', cate_T)
rS, mseS = score('S', cate_S)

# The T-learner should correlate strongly with the ground-truth effect.
assert rT > 0.7, f'T-learner correlation with true CATE too low: {rT:.3f}'
print('\nBoth learners recover the heterogeneity; T-learner corr > 0.7. OK.')

### 🔧 Exercise 2.1 — build the X-learner

The **X-learner** often beats the T-learner. Construction:

1. Use the already-fitted `m0`, `m1`.
2. **Impute** each unit's effect with the *other* arm's model:
   - treated units: `D1 = Y − m0.predict(X)`
   - control units: `D0 = m1.predict(X) − Y`
3. Regress `D1` on `X` (treated) and `D0` on `X` (control) to get `tau1`, `tau0`.
4. Blend by the propensity `e ≈ 0.5`: `cate_X = e*tau0.predict(Xte) + (1-e)*tau1.predict(Xte)`.

Fill in the `# TODO`s. The skeleton runs as-is (it falls back to the T-learner) so the notebook never breaks.

In [ ]:
e = 0.5   # known propensity (randomized)

# TODO 1: imputed effects using the OPPOSITE arm's model
D1 = ...   # for treated units:  Ytr[Wtr==1] - m0.predict(Xtr[Wtr==1])
D0 = ...   # for control units:  m1.predict(Xtr[Wtr==0]) - Ytr[Wtr==0]

# TODO 2: regress the imputed effects on X (use RandomForestRegressor(**RF))
# tau1 = RandomForestRegressor(**RF).fit(Xtr[Wtr==1], D1)
# tau0 = RandomForestRegressor(**RF).fit(Xtr[Wtr==0], D0)

# TODO 3: blend the two with the propensity e
# cate_X = e*tau0.predict(Xte) + (1-e)*tau1.predict(Xte)

# Fallback so the skeleton still runs before you fill it in:
if not isinstance(D1, np.ndarray):
    cate_X = cate_T.copy()
print('cate_X ready (fallback = T-learner until you complete the TODOs).')

### ✅ Solution 2.1

In [ ]:
D1 = Ytr[Wtr == 1] - m0.predict(Xtr[Wtr == 1])   # treated: effect vs control model
D0 = m1.predict(Xtr[Wtr == 0]) - Ytr[Wtr == 0]   # control: treated model vs actual

tau1 = RandomForestRegressor(**RF).fit(Xtr[Wtr == 1], D1)
tau0 = RandomForestRegressor(**RF).fit(Xtr[Wtr == 0], D0)

cate_X = e * tau0.predict(Xte) + (1 - e) * tau1.predict(Xte)

rX, mseX = score('X', cate_X)
print(f'\nT-learner MSE = {mseT:.3f}   X-learner MSE = {mseX:.3f}')
assert rX > 0.7, 'X-learner should also correlate strongly with truth'
assert mseX <= mseT + 0.02, 'X-learner should be at least competitive with T'
print('X-learner is competitive with (here, better than) the T-learner. OK.')

## 3 · Evaluating a CATE model — uplift / Qini

In real data τ(x) is unobserved, so you judge a model by whether **targeting by its score pays off.** Sort units by predicted CATE, then accumulate the (true, in this simulation) effect as you treat the top-ranked fraction. A useful model's curve bows **above** the random diagonal; the area between them is a Qini-style score.

In [ ]:
def uplift_curve(cate_pred, tau_true):
    order = np.argsort(-cate_pred)            # best-predicted first
    cum   = np.cumsum(tau_true[order])        # cumulative true gain
    frac  = np.arange(1, len(order) + 1) / len(order)
    rand  = np.linspace(0, cum[-1], len(order))  # treat in random order
    qini  = np.trapezoid(cum, frac) - np.trapezoid(rand, frac)
    return frac, cum, rand, qini

frac, cum, rand, qini_T = uplift_curve(cate_T, tau_te)
print(f'Qini-style area (T-learner, model - random) = {qini_T:.2f}')
assert qini_T > 0, 'a useful CATE model must beat random targeting'

fig, ax = plt.subplots()
ax.plot(frac, cum,  label='target by predicted CATE')
ax.plot(frac, rand, ls='--', label='random targeting')
ax.set_xlabel('fraction of units treated (high CATE first)')
ax.set_ylabel('cumulative true gain')
ax.set_title('Uplift / Qini curve — model bows above the diagonal')
ax.legend()
print('Curve rises fast then flattens: the early-treated units are the responders.')

The curve climbs steeply while we are treating high-CATE responders, peaks, then *declines* as we are forced to treat harmed units (negative τ). **The peak is the optimal treated fraction** — a preview of the policy in Section 4.

### 🔧 Exercise 3.1 — calibration by CATE quintile

A second check: **calibration.** Bin the test units into quintiles of predicted CATE and, within each bin, compare the *mean predicted* effect to the *mean true* effect. A well-calibrated model lands on the 45° line, so the binned predicted and true means should be highly correlated.

Complete the `# TODO`s using `pd.qcut(cate_T, 5, labels=False)`.

In [ ]:
# TODO: bin by predicted-CATE quintile and average pred & truth per bin.
q = ...   # pd.qcut(cate_T, 5, labels=False)
if not isinstance(q, np.ndarray):
    # fallback so the cell runs before you fill it in
    q = pd.qcut(cate_T, 5, labels=False)
cal = (pd.DataFrame({'pred': cate_T, 'true': tau_te, 'bin': q})
         .groupby('bin')[['pred', 'true']].mean())
print(cal)
# calib = ...   # corr between cal['pred'] and cal['true']

### ✅ Solution 3.1

In [ ]:
q = pd.qcut(cate_T, 5, labels=False)
cal = (pd.DataFrame({'pred': cate_T, 'true': tau_te, 'bin': q})
         .groupby('bin')[['pred', 'true']].mean())
calib = np.corrcoef(cal['pred'], cal['true'])[0, 1]
print(cal)
print(f'\nCalibration corr (binned pred vs. true) = {calib:.3f}')

fig, ax = plt.subplots()
lo = min(cal['pred'].min(), cal['true'].min())
hi = max(cal['pred'].max(), cal['true'].max())
ax.plot([lo, hi], [lo, hi], 'k--', lw=1, label='perfect calibration')
ax.scatter(cal['pred'], cal['true'], s=60, zorder=3)
ax.set_xlabel('mean predicted CATE in bin')
ax.set_ylabel('mean TRUE CATE in bin')
ax.set_title('Calibration: binned predicted vs. realized effect')
ax.legend()
assert calib > 0.9, 'binned predictions should track the truth closely'
print('Bins line up on the 45-degree line — well calibrated. OK.')

## 4 · From CATE to a policy — and its value

Now the payoff. Define the policy **π(x) = 1{τ̂(x) > 0}**: treat a unit only when its predicted effect is positive. The **value advantage** of any policy over treating no one is

$$V(\pi) - V(\text{none}) = \mathbb{E}[\pi(X)\,\tau(X)].$$

Since we know τ(x), we can compute this exactly and compare three policies: the learned rule, **treat-all**, and **treat-none**.

In [ ]:
def policy_value(pi, tau_true):
    """Expected gain over treat-none:  E[ pi(X) * tau(X) ]."""
    return np.mean(pi * tau_true)

pi_learned = (cate_T > 0).astype(int)          # treat if predicted CATE > 0
pi_all     = np.ones(len(Xte), dtype=int)       # treat everyone
pi_none    = np.zeros(len(Xte), dtype=int)      # treat no one
pi_oracle  = (tau_te > 0).astype(int)           # if we KNEW the truth

v_learned = policy_value(pi_learned, tau_te)
v_all     = policy_value(pi_all,     tau_te)
v_none    = policy_value(pi_none,    tau_te)
v_oracle  = policy_value(pi_oracle,  tau_te)

print('Policy value (gain over treat-none):')
print(f'  treat-none        : {v_none:+.3f}')
print(f'  treat-all         : {v_all:+.3f}   (net-negative: it treats the harmed)')
print(f'  learned 1{{CATE>0}} : {v_learned:+.3f}')
print(f'  oracle  1{{tau>0}}  : {v_oracle:+.3f}   (best achievable)')
print(f'\nLearned policy treats {pi_learned.mean():.1%} of units.')

assert v_learned > v_all,  'learned policy should beat treat-all'
assert v_learned > v_none, 'learned policy should beat treat-none'
print('\nThe learned policy beats BOTH treat-all and treat-none. OK.')

The learned rule **dominates both baselines**: treat-all is net-negative because it pays for the harmed units, treat-none gains nothing, and our 'treat if τ̂ > 0' rule captures most of the oracle's value while treating only those it expects to help. *That* is the point of estimating CATE — it changes the decision.

### 🔧 Exercise 4.1 — a budgeted policy

Often you can only treat a **fraction** of units (a budget). The budget-`b` policy treats the top-`b` share by predicted CATE. Using `cate_T`, build the policy that treats the **top 40%** and compute its value advantage over treat-none. Does it beat treat-all?

Fill in the `# TODO`s.

In [ ]:
budget = 0.40
# TODO: threshold cate_T at its (1 - budget) quantile to pick the top 40%.
thresh = ...   # np.quantile(cate_T, 1 - budget)
if thresh is ...:
    thresh = np.quantile(cate_T, 1 - budget)
pi_budget = (cate_T >= thresh).astype(int)
# v_budget = ...   # policy_value(pi_budget, tau_te)
print('fraction treated:', round(pi_budget.mean(), 3))

### ✅ Solution 4.1

In [ ]:
thresh = np.quantile(cate_T, 1 - budget)
pi_budget = (cate_T >= thresh).astype(int)
v_budget = policy_value(pi_budget, tau_te)
print(f'budget-{budget:.0%} policy treats {pi_budget.mean():.1%} of units')
print(f'  value over treat-none = {v_budget:+.3f}')
print(f'  (treat-all = {v_all:+.3f}, learned-all-positive = {v_learned:+.3f})')
assert v_budget > v_all, 'targeting the top 40% should beat treating everyone'
print('Even under a tight budget, targeting by CATE beats treat-all. OK.')

## 5 · Wrap-up & self-check

- **CATE τ(x) = E[Y(1) − Y(0) | X = x]**; the ATE is its average, so a near-zero ATE can hide large, canceling individual effects.
- **Meta-learners** turn any regressor into a CATE estimator: S (one model), T (two models), X (cross-impute + blend). Here the T- and X-learners recovered τ(x) with correlation ≈ 0.9.
- A **causal forest** would add honest splitting and valid pointwise confidence intervals — the T-learner forest is our inference-free approximation.
- You can evaluate CATE **without ground truth** via **calibration** (binned pred vs. realized) and an **uplift / Qini curve** (does targeting pay?). Naive subgroup-hunting overfits — always use a holdout.
- A CATE estimate becomes a decision: **π(x) = 1{τ̂(x) > c}**, with value advantage **E[π(X)(τ(X) − c)]**. The learned policy beat both treat-all and treat-none.

**You're ready for Week 14 — Advanced topics**, where we add mediation, sensitivity analysis for unmeasured confounding, and effects under interference to the toolkit.